CREATING A SYNTHETIC DATASET FOR MY KOPASCORE PROJECT.

In [1]:
import numpy as np
import pandas as pd

In [2]:
#This ensures numpy will create the same random numbers so that results will match
np.random.seed(42)

In [3]:
#Just defining the number of users
num_users = 10000

In [4]:
#Here now I am generating user IDs
#Just creating a list
user_ids = [f"USR_{i}" for i in range (10000, 10000+num_users)]

In [5]:
#here i generate monthly income
monthly_inflow = np.random.normal(
    loc = 35000, #mean
    scale = 15000, #standard deviation
    size = num_users
)

monthly_inflow = np.clip(monthly_inflow, 0, None) #ensures that there are no negatives

In [6]:
#generating income volatility - this is how much your income fluctuates
#ranges from 5% to 60%
inflow_volatility = np.random.uniform(
    low = 0.05,
    high = 0.6,
    size = num_users
)

In [7]:
#generating outflows here (Spending)
#people usually spend a percentage of what they earn (60% to 110%)
spending_ratio = np.random.uniform(
    low = 0.6,
    high = 1.10,
    size = num_users
    )

monthly_outflow = monthly_inflow * spending_ratio

In [8]:
#Generating airtime top-up count
#random integer between 2 and 30
airtime_count = np.random.randint(
    low = 2,
    high = 30,
    size = num_users
)

In [9]:
#generating historical defaults
#poisson distribution gives realistic amounts centered around 0.4 defaults
prior_defaults = np.random.poisson(
    lam = 0.4,
    size = num_users
)

In [15]:
#generating loan request amounts
loan_amount = np.random.normal(
    loc = 15000,
    scale = 10000,
    size = num_users
)

loan_amount=np.clip(
    loan_amount,
    0,
    None
)

In [16]:
#cREATE THE DATAFRAME
df = pd.DataFrame(
    {
    "user_id": user_ids,
    "monthly_inflow": monthly_inflow,
    "inflow_volatility": inflow_volatility,
    "monthly_outflow": monthly_outflow,
    "airtime_count": airtime_count,
    "prior_defaults": prior_defaults,
    "loan_amount": loan_amount
    }
)

df.head()

,user_id,monthly_inflow,inflow_volatility,monthly_outflow,airtime_count,prior_defaults,loan_amount
0,USR_10000,42450.712295,0.226690,29110.784453,16,0,26826.644955
1,USR_10001,32926.035482,0.106661,25465.032387,13,0,13883.329551
2,USR_10002,44715.328072,0.089494,48518.406170,9,0,19618.615284
3,USR_10003,57845.447846,0.101852,57289.780019,17,1,23037.374540
4,USR_10004,31487.699379,0.370578,23341.687219,13,1,14290.947637


In [27]:
# Replace 0 in outflows or inflows with a tiny number (1.0 KES) to avoid dividing by 0
df["monthly_outflow"] = df["monthly_outflow"].replace(0, 1.0)
df["monthly_inflow"] = df["monthly_inflow"].replace(0, 1.0)

Now I'm going to do some feature Enginnering

In [28]:
df["net_cashflow_ratio"] = np.round(df["monthly_inflow"]/df["monthly_outflow"],3)
#This one measures how much money comes in compared to how much money goes out.


This one measures how much money comes in compared to how much money goes out.

Above 1.0 --> the guy is saving some money

Exactly 1.0 --> brother is living from paycheck to paycheck

below 1.0 --> guy is spending more than he should

This is a great indicator especially when it comes to loans cause you can be able to see who will struggle and who won't

In [29]:
df["loan_to_income_ratio"] = np.round(df["loan_amount"]/df["monthly_inflow"],3)

This one measures how big the requested loan is relative to the guy's monthly income.

Low ratio (0.1, 0.2) --> The loan is small compared to your earnings so that's a low risk

Higth ratio (0.7, 0.8) --> The loan is almost the exact amount to your earnings so you're going to struggle paying it. That's a high  risk

In [30]:
df.head()

,user_id,monthly_inflow,inflow_volatility,monthly_outflow,airtime_count,prior_defaults,loan_amount,net_cashflow_ratio,loan_to_income_ratio
0,USR_10000,42450.712295,0.226690,29110.784453,16,0,26826.644955,1.458,0.632
1,USR_10001,32926.035482,0.106661,25465.032387,13,0,13883.329551,1.293,0.422
2,USR_10002,44715.328072,0.089494,48518.406170,9,0,19618.615284,0.922,0.439
3,USR_10003,57845.447846,0.101852,57289.780019,17,1,23037.374540,1.010,0.398
4,USR_10004,31487.699379,0.370578,23341.687219,13,1,14290.947637,1.349,0.454


Now I am going to create the target column, but I first have to create some rules

First the basic risk score equation

In [31]:
risk_score = (
    (df["loan_to_income_ratio"]*3.0) +
     ((1.5 - df["net_cashflow_ratio"])*2.0) +
      (df["inflow_volatility"]*4.0) +
       (df["prior_defaults"]*1.5)
)

So here we have done a liitle bit of feature selection, taking the 4 financial key red flags into a single number called risk_score

    (df["loan_to_income_ratio"]*3.0)
If someone asks for a loan that is huge compared to their salary, their risk jumps up fast. We multiply by 3.0 to give it a strong weight.

      ((1.5 - df["net_cashflow_ratio"])*2.0)

If net_cashflow_ratio is low (e.g., 0.8, meaning spending exceeds income), 1.5 - 0.8 = 0.7 (a positive penalty). If they save lots of money (e.g., 1.4), 1.5 - 1.4 = 0.1 (almost zero penalty).

      (df["inflow_volatility"]*4.0) +

Unpredictable income (gig work, irregular payments) makes paying back loans on time harder. We weight this heavily (4.0).

       (df["prior_defaults"]*1.5)
Past behavior predicts future behavior! Each loan they previously failed to repay adds 1.5 directly to their risk score.

In [32]:
default_probability = 1 / (1+np.exp(-risk_score+3.0))

my calculated risk score could be any number so we can convert it to a probability using the sigmoid function. Sigmoid function is best because it squashes any number, whether positive or negative between 0 and 1

Then the +3.0 shifts the curve so that average borrowers sit around a low 10% probability of default, matching real-world credit data.

In [34]:
default_probability = np.clip(default_probability, 0, 1)
df["is_default"] = np.random.binomial(n=1, p=default_probability, size=num_users)

In [35]:
df.isna().sum()

,0
user_id,0
monthly_inflow,0
inflow_volatility,0
monthly_outflow,0
airtime_count,0
prior_defaults,0
loan_amount,0
net_cashflow_ratio,0
loan_to_income_ratio,0
is_default,0


In [37]:
df.head(20)

,user_id,monthly_inflow,inflow_volatility,monthly_outflow,airtime_count,prior_defaults,loan_amount,net_cashflow_ratio,loan_to_income_ratio,is_default
0,USR_10000,42450.712295,0.226690,29110.784453,16,0,26826.644955,1.458,0.632,0
1,USR_10001,32926.035482,0.106661,25465.032387,13,0,13883.329551,1.293,0.422,0
2,USR_10002,44715.328072,0.089494,48518.406170,9,0,19618.615284,0.922,0.439,0
3,USR_10003,57845.447846,0.101852,57289.780019,17,1,23037.374540,1.010,0.398,1
4,USR_10004,31487.699379,0.370578,23341.687219,13,1,14290.947637,1.349,0.454,1
5,USR_10005,31487.945646,0.078414,27771.451282,25,0,21809.690625,1.134,0.693,0
6,USR_10006,58688.192233,0.235310,47077.499951,26,0,22672.069326,1.247,0.386,1
7,USR_10007,46511.520937,0.464260,39301.425341,28,1,20311.325042,1.183,0.437,1
8,USR_10008,27957.884211,0.211891,21739.578154,7,0,23101.337391,1.286,0.826,1
9,USR_10009,43138.400654,0.469819,31907.987088,24,0,35683.290232,1.352,0.827,1
